# Criação e Validação do Modelo Preditivo (Machine Learning)Objetivo: Construir um modelo preditivo que classifique a probabilidade do aluno ou aluna entrar em risco de defasagem (adequação ao nível escolar - IAN).Modelos a testar: Regressão Logística, Random Forest e XGBoost.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-Learn e XGBoost
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
import pickle


## 1. Carga de Dados

In [ ]:
df = pd.read_csv('dados_limpos_pm.csv')
df.head()


## 2. Feature EngineeringA variável alvo (target) será `risco_defasagem` que criamos na fase de ETL. Ela vale `1` se a adequação (IAN) estiver nos níveis Moderado ou Severo (<= 6.9) ou Severo (<= 5). A regra atual do CSV está (IAN <= 5). Vamos conferir o balanceamento.

In [ ]:
print(df['risco_defasagem'].value_counts())
sns.countplot(data=df, x='risco_defasagem')
plt.title('Distribuição da Variável Target (Risco Defasagem)')
plt.show()


### Tratamento de Categóricas (Label Encoding / One-Hot)

In [ ]:
# Features a utilizar: indicadores acadêmicos/psicológicos, idade de ingresso, gênero, status de bolsa
# Não vamos usar o IAN no treino, pois o risco é extraído diretamente dele (Data Leakage).
# Removemos RA (índice), Pedra, Fases em Texto e Ano para simplificar.

cols_features = ['idade_unificada', 'genero', 'bolsa', 'ida', 'ieg', 'iaa', 'ips', 'ipp', 'ipv']

X = df[cols_features].copy()
y = df['risco_defasagem'].copy()

# Encode categóricas
le_genero = LabelEncoder()
X['genero'] = le_genero.fit_transform(X['genero'])

le_bolsa = LabelEncoder()
X['bolsa'] = le_bolsa.fit_transform(X['bolsa'])

# Lidando com valores nulos caso haja nas features
X.fillna(X.median(), inplace=True)

X.head()


## 3. Divisão Treino e Teste (OOT ou Random)

In [ ]:
# Seed fixa para reproducibilidade
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Tamanho Treino: {X_train.shape[0]} | Tamanho Teste: {X_test.shape[0]}")

# Escalonamento apenas para Regressão Logística (Árvores não precisam, mas padronizaremos por segurança)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 4. Treinamento de ModelosVamos testar 3 enfoques: 1 Linear (Regressão Logística) e 2 Baseados em Árvores (Random Forest e XGBoost).

In [ ]:
## Regressão Logística
logreg = LogisticRegression(random_state=42, class_weight='balanced')
logreg.fit(X_train_scaled, y_train)
y_pred_log = logreg.predict(X_test_scaled)
y_proba_log = logreg.predict_proba(X_test_scaled)[:, 1]

## Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

## XGBoost
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', max_depth=4, learning_rate=0.05, n_estimators=150, random_state=42, scale_pos_weight=(len(y_train)-sum(y_train))/sum(y_train))
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]


## 5. Avaliação e Comparação

In [ ]:
modelos = ['Regressão Logística', 'Random Forest', 'XGBoost']
probs = [y_proba_log, y_proba_rf, y_proba_xgb]
preds = [y_pred_log, y_pred_rf, y_pred_xgb]

plt.figure(figsize=(10, 8))
for nome, prob in zip(modelos, probs):
    auc = roc_auc_score(y_test, prob)
    fpr, tpr, _ = roc_curve(y_test, prob)
    plt.plot(fpr, tpr, label=f"{nome} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], 'k--')
plt.title('Curva ROC - Comparativo de Modelos')
plt.xlabel('Taxa de Falsos Positivos')
plt.ylabel('Taxa de Verdadeiros Positivos')
plt.legend()
plt.show()


In [ ]:
print("==== RANDOM FOREST ====\n", classification_report(y_test, y_pred_rf))
print("==== XGBOOST ====\n", classification_report(y_test, y_pred_xgb))
print("==== REGRESSÃO LOGÍSTICA ====\n", classification_report(y_test, y_pred_log))


### Importância das Variáveis (Feature Importance) no Random Forest / XGBoost

In [ ]:
importances = pd.DataFrame({'Feature': cols_features, 'Importância': rf.feature_importances_}).sort_values(by='Importância', ascending=False)
sns.barplot(data=importances, x='Importância', y='Feature', palette='magma')
plt.title('Importância das Variáveis para o Risco Preditivo (Random Forest)')
plt.show()


## 6. Salvar o Melhor Modelo para a Aplicação StreamlitAssumindo que decidiremos pelo modelo de Árvore baseado na métrica (ex: XGBoost ou Random Forest), exportamos o objeto modelo treinado.

In [ ]:
# Exportando o modelo Random Forest como exemplo padrão se ele performar bem.
# Além do modelo em si, precisamos de persistir Label Encoders se eles forem ser usados.
modelo_final = {
    'modelo': rf, # Pode trocar para xgb_model caso queira Deploy do XGBoost
    'le_genero': le_genero,
    'le_bolsa': le_bolsa,
    'colunas': cols_features
}

with open('modelo_risco.pkl', 'wb') as f:
    pickle.dump(modelo_final, f)

print("Modelo 'modelo_risco.pkl' exportado e pronto para deploy no Streamlit!")
